In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.feature_selection import VarianceThreshold


# here i am selecting featues and reducing them 

In [2]:
import seaborn as sns
df=pd.read_csv('../data//processed/secom_data.csv')

sensor_col=[col for col in df.columns if 'sensor' in col]

print(df.shape)
print(len(sensor_col))


(1567, 592)
590


In [3]:
missing_pct = (df[sensor_col].isnull().sum() / len(df)) * 100

high_missing = missing_pct[missing_pct > 50].index.tolist()

df_clean = df.drop(columns=high_missing)

sensor_cols_clean = [col for col in df_clean.columns if 'sensor' in col]

print(f"Before : {len(sensor_col)} sensors")
print(f"Dropped: {len(high_missing)} sensors")
print(f"After  : {len(sensor_cols_clean)} sensors")

Before : 590 sensors
Dropped: 28 sensors
After  : 562 sensors


In [4]:
from sklearn.impute import SimpleImputer


#mujhe threshold se neecher varaiance wali features ko remove krna hai uske liye temp_o+imputer bnake usse median se replace krdia tabhi nan na aaye variance ke liye 
temp_imputer=SimpleImputer(strategy='median')
X_temp=pd.DataFrame(temp_imputer.fit_transform(df_clean[sensor_cols_clean]),columns=sensor_cols_clean)



variance=X_temp.var()

threshold=0.01

low_variance=variance[variance<threshold].index.tolist()

X_var=X_temp.drop(columns=low_variance)

print(f"Before : {len(sensor_cols_clean)} sensors")
print(f"Dropped: {len(low_variance)} sensors")
print(f"After  : {X_var.shape[1]} sensors")

Before : 562 sensors
Dropped: 265 sensors
After  : 297 sensors


In [5]:
corr_matrix=X_var.corr().abs()

In [6]:
corr_matrix

,sensor_0,sensor_1,sensor_2,sensor_3,sensor_4,sensor_6,sensor_12,sensor_14,sensor_15,sensor_16,...,sensor_569,sensor_570,sensor_571,sensor_572,sensor_573,sensor_574,sensor_576,sensor_577,sensor_585,sensor_589
sensor_0,1.000000,0.144161,0.004667,0.006665,0.010819,0.002028,0.010571,0.007191,0.030745,0.005720,...,0.052853,0.018987,0.023194,0.013715,0.002035,0.015242,0.013265,0.008639,0.023695,0.004185
sensor_1,0.144161,1.000000,0.005883,0.008963,0.001917,0.025222,0.034308,0.037730,0.087226,0.001812,...,0.026826,0.008994,0.037897,0.001705,0.011468,0.001255,0.002522,0.010156,0.002231,0.044552
sensor_2,0.004667,0.005883,1.000000,0.298810,0.095881,0.136212,0.018359,0.006494,0.006116,0.000805,...,0.061432,0.037054,0.015672,0.000554,0.030714,0.001305,0.002555,0.028737,0.015749,0.032763
sensor_3,0.006665,0.008963,0.298810,1.000000,0.058351,0.685773,0.028949,0.020144,0.013440,0.004475,...,0.060694,0.002004,0.016797,0.008152,0.013742,0.007345,0.008741,0.016888,0.025882,0.080942
sensor_4,0.010819,0.001917,0.095881,0.058351,1.000000,0.074395,0.002800,0.017583,0.011395,0.001730,...,0.031196,0.005227,0.081776,0.011918,0.016415,0.012158,0.012056,0.003978,0.001645,0.050792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
sensor_574,0.015242,0.001255,0.001305,0.007345,0.012158,0.007650,0.032951,0.000415,0.024009,0.013994,...,0.014780,0.307529,0.138441,0.993689,0.781319,1.000000,0.991738,0.851784,0.016837,0.020427
sensor_576,0.013265,0.002522,0.002555,0.008741,0.012056,0.007275,0.035784,0.000978,0.023486,0.014156,...,0.011877,0.360498,0.136232,0.994772,0.790026,0.991738,1.000000,0.859278,0.017176,0.022523
sensor_577,0.008639,0.010156,0.028737,0.016888,0.003978,0.012458,0.031385,0.009500,0.019169,0.004404,...,0.002534,0.247655,0.121115,0.863768,0.957874,0.851784,0.859278,1.000000,0.023986,0.024740
sensor_585,0.023695,0.002231,0.015749,0.025882,0.001645,0.039545,0.000534,0.002597,0.017797,0.002641,...,0.047843,0.010523,0.006642,0.017199,0.022770,0.016837,0.017176,0.023986,1.000000,0.003722


In [7]:
#upper traingle lo(to avoid duplicate )

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# >0.95 correlated features drop karo
threshold_corr = 0.95
to_drop_corr = [col for col in upper.columns
                if any(upper[col] > threshold_corr)]

X_filtered = X_var.drop(columns=to_drop_corr)

print(f"Before : {X_var.shape[1]} sensors")
print(f"Dropped: {len(to_drop_corr)} sensors")
print(f"After  : {X_filtered.shape[1]} sensors")

Before : 297 sensors
Dropped: 102 sensors
After  : 195 sensors


In [8]:
X_filtered['label'] = df['labels'].values

In [9]:
X_filtered

,sensor_0,sensor_1,sensor_2,sensor_3,sensor_4,sensor_6,sensor_12,sensor_14,sensor_15,sensor_16,...,sensor_563,sensor_564,sensor_569,sensor_570,sensor_571,sensor_572,sensor_573,sensor_585,sensor_589,label
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,97.6133,202.4396,7.9558,414.8710,10.04330,...,0.6510,5.16,16.98835,533.8500,2.1113,8.95,0.3157,2.3630,71.9005,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,102.3433,200.5470,10.1548,414.7347,9.25990,...,0.6510,5.16,16.98835,535.0164,2.4335,5.92,0.2653,4.4447,208.2045,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,95.4878,202.0179,9.5157,416.7075,9.31440,...,0.9032,1.10,68.84890,535.0245,2.0293,11.21,0.1882,3.1745,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,104.2367,201.8482,9.6052,422.2894,9.69240,...,0.6511,7.32,25.03630,530.5682,2.0253,9.33,0.1738,2.0544,73.8432,-1
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.3967,201.9424,10.5661,420.5925,10.33870,...,0.6510,5.16,16.98835,532.0155,2.0275,8.83,0.2224,99.3032,73.8432,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2899.41,2464.36,2179.7333,3085.3781,1.4843,82.2467,203.9867,11.7692,419.3404,10.23970,...,0.5671,4.98,15.46620,536.3418,2.0153,7.98,0.2363,2.8669,203.1720,-1
1563,3052.31,2522.55,2198.5667,1124.6595,0.8763,98.4689,204.0173,9.1620,405.8178,10.22850,...,0.6254,4.56,20.91180,537.9264,2.1814,5.48,0.3891,2.6238,203.1720,-1
1564,2978.81,2379.78,2206.3000,1110.4967,0.8236,99.4122,199.5356,8.9670,412.2191,9.85175,...,0.8209,11.09,29.09540,530.3709,2.3435,6.49,0.4154,3.0590,43.5231,-1
1565,2894.92,2532.01,2177.0333,1183.7287,1.5726,98.7978,197.2448,9.7354,401.9153,9.86300,...,0.5671,4.98,15.46620,534.3936,1.9098,9.13,0.3669,3.5662,93.4941,-1


In [10]:
X_filtered.to_csv('../data/processed/secom_features.csv', index=False)


In [11]:
#what is smote

""" Tumhare paas 1000 PASS chips hain aur sirf 70 FAIL chips.
Model sirf PASS seekhega — FAIL ignore karega!
SMOTE = Synthetic Minority Oversampling TEchnique
Existing FAIL samples ke beech mein naye synthetic FAIL samples banata hai! """

' Tumhare paas 1000 PASS chips hain aur sirf 70 FAIL chips.\nModel sirf PASS seekhega — FAIL ignore karega!\nSMOTE = Synthetic Minority Oversampling TEchnique\nExisting FAIL samples ke beech mein naye synthetic FAIL samples banata hai! '

In [12]:
""" Original FAIL samples:
Point A = [sensor_1: 3.2, sensor_2: 1.5]
Point B = [sensor_1: 4.1, sensor_2: 2.3]

SMOTE ek synthetic point banata hai beech mein:
Point C = [sensor_1: 3.6, sensor_2: 1.9]
          (A aur B ke beech interpolation) """

' Original FAIL samples:\nPoint A = [sensor_1: 3.2, sensor_2: 1.5]\nPoint B = [sensor_1: 4.1, sensor_2: 2.3]\n\nSMOTE ek synthetic point banata hai beech mein:\nPoint C = [sensor_1: 3.6, sensor_2: 1.9]\n          (A aur B ke beech interpolation) '

In [13]:

X=X_filtered.copy()
y=df['labels'].map({-1:1,1:0})

""" -1->1(Pass) else Fail
 """

' -1->1(Pass) else Fail\n '

In [14]:
X['label'].value_counts()

label
-1    1463
 1     104
Name: count, dtype: int64

In [15]:
y.value_counts()

labels
1    1463
0     104
Name: count, dtype: int64

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [17]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [18]:
preprocessing_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

In [19]:
X_train_scaled=preprocessing_pipe.fit_transform(X_train)
X_test_scaled=preprocessing_pipe.transform(X_test)

smote=SMOTE(random_state=42,k_neighbors=5)

X_train_res,y_train_res=smote.fit_resample(X_train_scaled,y_train)


In [20]:
print("=" * 40)
print("   SMOTE RESULTS")
print("=" * 40)
print(f"Before SMOTE:")
print(f"  PASS: {(y_train==0).sum()}")
print(f"  FAIL: {(y_train==1).sum()}")
print(f"\nAfter SMOTE:")
print(f"  PASS: {(y_train_res==0).sum()}")
print(f"  FAIL: {(y_train_res==1).sum()}")
print(f"\nTotal samples: {len(y_train_res)}")
print("=" * 40)

   SMOTE RESULTS
Before SMOTE:
  PASS: 83
  FAIL: 1170

After SMOTE:
  PASS: 1170
  FAIL: 1170

Total samples: 2340


In [21]:
""" 
> Socho ek **school exam** hai jisme:
> - **1170 students pass** hue
> - **83 students fail** hue
>
> Teacher ko ML model banana hai jo predict kare — kaun fail hoga.
>
> **Problem:** Model ne socha —
> *"Seedha sabko PASS bol deta hoon — 93% accuracy!"* 🤦
>
> Yeh **cheating** hai — useless model!

---

> **SMOTE ka kaam:**
>
> Teacher ke paas sirf 83 fail students hain.
> Woh 83 fail students ko **dhyan se dekha** —
> unke features dekhe (kitna padhte the, attendance, etc.)
>
> Phir **similar naye fake students banaye** jo fail jaise lagte hain — jo real fail students ke beech mein hote!
>
> ```
> Real Fail Student A:  [attendance: 30%, marks: 25]
> Real Fail Student B:  [attendance: 40%, marks: 30]
>
> SMOTE ne banaya:
> Fake Fail Student C:  [attendance: 35%, marks: 27]
>                        ↑ A aur B ke beech mein!
> ```
>
> Ab teacher ke paas **1170 fail students** hain — model theek se seekh sakta hai!

---
 """

' \n> Socho ek **school exam** hai jisme:\n> - **1170 students pass** hue\n> - **83 students fail** hue\n>\n> Teacher ko ML model banana hai jo predict kare — kaun fail hoga.\n>\n> **Problem:** Model ne socha —\n> *"Seedha sabko PASS bol deta hoon — 93% accuracy!"* 🤦\n>\n> Yeh **cheating** hai — useless model!\n\n---\n\n> **SMOTE ka kaam:**\n>\n> Teacher ke paas sirf 83 fail students hain.\n> Woh 83 fail students ko **dhyan se dekha** —\n> unke features dekhe (kitna padhte the, attendance, etc.)\n>\n> Phir **similar naye fake students banaye** jo fail jaise lagte hain — jo real fail students ke beech mein hote!\n>\n> ```\n> Real Fail Student A:  [attendance: 30%, marks: 25]\n> Real Fail Student B:  [attendance: 40%, marks: 30]\n>\n> SMOTE ne banaya:\n> Fake Fail Student C:  [attendance: 35%, marks: 27]\n>                        ↑ A aur B ke beech mein!\n> ```\n>\n> Ab teacher ke paas **1170 fail students** hain — model theek se seekh sakta hai!\n\n---\n '

In [22]:
np.save('../data/processed/X_train.npy', X_train_res)
np.save('../data/processed/X_test.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train_res)
np.save('../data/processed/y_test.npy', y_test)

In [23]:
""" NPY:
Binary format — human readable nahi
Seedha numbers memory mein save hote hain """

' NPY:\nBinary format — human readable nahi\nSeedha numbers memory mein save hote hain '

In [24]:
print(X_filtered.shape)
print(X_filtered.columns.tolist()[-5:])

(1567, 196)
['sensor_572', 'sensor_573', 'sensor_585', 'sensor_589', 'label']


In [25]:
# Label hatao X se
X = X_filtered.drop(columns=['label'])
y = df['labels'].map({-1: 0, 1: 1})

print(f"X shape: {X.shape}")  # (1567, 195)
print(f"y shape: {y.shape}")  # (1567,)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Pipeline
preprocessing_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

X_train_scaled = preprocessing_pipeline.fit_transform(X_train)
X_test_scaled = preprocessing_pipeline.transform(X_test)

# SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train
)

print(f"\n✅ X_train: {X_train_resampled.shape}")
print(f"✅ X_test : {X_test_scaled.shape}")
print(f"✅ y_train: {y_train_resampled.shape}")
print(f"✅ y_test : {y_test.shape}")

# Save karo
np.save('../data/processed/X_train.npy', X_train_resampled)
np.save('../data/processed/X_test.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train_resampled)
np.save('../data/processed/y_test.npy', y_test)
print("\n✅ Saved!")

X shape: (1567, 195)
y shape: (1567,)

✅ X_train: (2340, 195)
✅ X_test : (314, 195)
✅ y_train: (2340,)
✅ y_test : (314,)

✅ Saved!


In [26]:
# Check karo SMOTE se pehle kya available hai
print("X_train_scaled:", X_train_scaled.shape)
print("y_train:", y_train.shape)
print("X_val available?")

X_train_scaled: (1253, 195)
y_train: (1253,)
X_val available?


In [27]:
# Val split SMOTE se pehle!
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"X_tr  : {X_tr.shape}")
print(f"X_val : {X_val.shape}")

# Sirf X_tr pe SMOTE!
smote = SMOTE(random_state=42, k_neighbors=5)
X_tr_smote, y_tr_smote = smote.fit_resample(X_tr, y_tr)

print(f"\nAfter SMOTE:")
print(f"X_tr_smote : {X_tr_smote.shape}")
print(f"PASS: {(y_tr_smote==0).sum()}")
print(f"FAIL: {(y_tr_smote==1).sum()}")

# Save karo
np.save('../data/processed/X_tr_smote.npy', X_tr_smote)
np.save('../data/processed/X_val.npy', X_val)
np.save('../data/processed/y_tr_smote.npy', y_tr_smote)
np.save('../data/processed/y_val.npy', y_val)

print("\n✅ Saved!")

X_tr  : (1002, 195)
X_val : (251, 195)

After SMOTE:
X_tr_smote : (1872, 195)
PASS: 936
FAIL: 936

✅ Saved!


In [28]:
print("X_tr_smote shape:", X_tr_smote.shape)
print("PASS:", (y_tr_smote==0).sum())
print("FAIL:", (y_tr_smote==1).sum())

X_tr_smote shape: (1872, 195)
PASS: 936
FAIL: 936
